# Job Scout — Phase 1 Walkthrough

**Build → Evaluate → Self-Improve.** This is Phase 1: a working, fully-traced
job-matching agent. This notebook is a guided runner — verify your environment,
run the agent on a fixture CV, and explore the Opik trace.

**Learning materials:**
- [LangGraph](https://langchain-ai.github.io/langgraph/) · [Opik](https://www.comet.com/docs/opik/) · [Gradio](https://www.gradio.app/) · [Adzuna API](https://developer.adzuna.com/)

> The agent's LLM output is *deliberately unoptimized* in Phase 1. We measure it in Phase 2 and improve it in Phase 3.

## 1. Environment check

Confirm the project imports and the key dependencies are installed.

In [ ]:
import importlib.metadata as m

for pkg in ["langgraph", "langchain", "opik", "gradio", "pypdf", "pydantic"]:
    print(f"{pkg:12s} {m.version(pkg)}")

from job_scout.config import get_settings
settings = get_settings()
print("\nmodel:", settings.scout_model)
print("opik enabled:", settings.has_opik)
print("adzuna keys:", settings.has_adzuna)

## 2. Service / key checks

The app runs with **zero keys** (Remotive + cached jobs, no tracing). For the full
lesson set `OPENAI_API_KEY`, `OPIK_API_KEY` / `OPIK_WORKSPACE`, and optionally
`ADZUNA_APP_ID` / `ADZUNA_APP_KEY` in `.env`.

In [ ]:
checks = {
    "LLM key (OPENAI_API_KEY)": bool(settings.openai_api_key.get_secret_value()),
    "Opik tracing": settings.has_opik,
    "Adzuna (live intl jobs)": settings.has_adzuna,
}
for name, ok in checks.items():
    print(f"[{'x' if ok else ' '}] {name}")

## 3. Read a fixture CV

Four synthetic CVs ship in `data/fixture_cvs/`. Extraction is text-only (pypdf).

In [ ]:
from pathlib import Path
from job_scout.tools.cv_reader import extract_cv_text

cv_path = Path("../data/fixture_cvs/senior_mle_eu.pdf")
cv_text = extract_cv_text(cv_path)
print(cv_text[:400])

## 4. Run the agent

One compiled graph: extract profile → fetch jobs (LLM picks the search args) → rank → maybe reformulate. We stream node-level status, exactly as the app does.

In [ ]:
from uuid import uuid4
from job_scout.runner import stream_run

result = None
for kind, payload in stream_run(cv_text, cv_path=str(cv_path), thread_id=str(uuid4()), tags=["phase-1", "notebook"]):
    if kind == "status":
        print("…", payload)
    else:
        result = payload

print("\nfailed:", result.failed)
print("jobs fetched:", result.n_jobs_fetched, "| ranked:", result.n_jobs_ranked)
print("sources:", result.jobs_sources, "| reformulations:", result.reformulation_count)
print(f"cost ${result.cost_usd:.4f} · {result.latency_s}s")

## 5. Explore the profile and rankings

In [ ]:
p = result.profile
if p:
    print(f"{p.name} · {p.seniority} · {', '.join(p.primary_roles)}")
    print("skills:", ", ".join(p.skills))
print()
for r in result.ranked_jobs[:5]:
    print(f"[{r.fit_score:3d}] {r.job.title} @ {r.job.company} ({r.job.location})")
    print("     ", r.fit_explanation[:160])

## 6. Explore the trace in Opik

If tracing is enabled, open your `job-scout` project in Opik. You should see:

- the **span tree** per node (extract_profile → fetch_jobs → rank_jobs → …),
- **Show Agent Graph** working in the sidebar,
- **per-run cost** (for OpenAI models),
- the **CV PDF attached** to the trace.

In [ ]:
from job_scout.tracing import opik_url
print("Opik project:", opik_url())

## 7. Baseline batch

`scripts/run_batch.py` runs ~50 cases (fixture CVs × locations + hard cases),
tags traces `baseline-batch`, and writes `reports/baseline.json`. Its failures
and weaknesses are documented in `reports/phase1_findings.md` — **and left
unfixed** (that is Phase 3's job).

```bash
uv run python scripts/run_batch.py --yes
```

**Next:** Phase 2 adds the tailoring node and the full evaluation stack.